In [6]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D


# ==============================
# 2. LOAD DATA
# ==============================
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_csv(url, sep="\t", names=["label", "message"])

print(df.head())
print(df["label"].value_counts())


# ==============================
# 3. CONVERT LABELS
# ==============================
df["label"] = df["label"].map({"ham": 0, "spam": 1})


# ==============================
# 4. TRAIN TEST SPLIT
# ==============================
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))


# ==============================
# 5. TOKENIZATION
# ==============================
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print("\nExample text:")
print(X_train.iloc[0])

print("\nConverted to numbers:")
print(X_train_seq[0])


# ==============================
# 6. PADDING
# ==============================
max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

print("\nShape:", X_train_pad.shape)


# ==============================
# 7. BUILD MODEL
# ==============================
model = Sequential([
    Embedding(input_dim=5000, output_dim=32, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.summary()


# ==============================
# 8. COMPILE MODEL
# ==============================
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# ==============================
# 9. TRAIN MODEL
# ==============================
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    validation_data=(X_test_pad, y_test)
)


# ==============================
# 10. EVALUATE MODEL
# ==============================
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("\nTest Accuracy:", accuracy)


# ==============================
# 11. PREDICTIONS
# ==============================
pred_probs = model.predict(X_test_pad)
preds = (pred_probs > 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))

print("\nClassification Report:")
print(classification_report(y_test, preds))

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64
Train size: 4457
Test size : 1115

Example text:
Reply to win £100 weekly! Where will the 2006 FIFA World Cup be held? Send STOP to 87239 to end service

Converted to numbers:
[87, 2, 172, 485, 461, 121, 33, 5, 3857, 3858, 315, 1268, 30, 2603, 67, 82, 2, 1657, 2, 316, 238]

Shape: (4457, 100)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.8604 - loss: 0.3805 - val_accuracy: 0.8664 - val_loss: 0.3631
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8658 - loss: 0.3546 - val_accuracy: 0.8664 - val_loss: 0.3472
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8658 - loss: 0.3262 - val_accuracy: 0.8664 - val_loss: 0.3076
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8883 - loss: 0.2405 - val_accuracy: 0.9372 - val_loss: 0.1775
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9589 - loss: 0.1334 - val_accuracy: 0.9659 - val_loss: 0.1037
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9659 - loss: 0.1037

Test Accuracy: 0.9659192562103271
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

Confusion Matrix:
[[964   2]
 [ 36 113]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       0.98      0.76      0.8

In [7]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D, Conv1D, MaxPooling1D


# ==============================
# 2. LOAD DATA
# ==============================
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_csv(url, sep="\t", names=["label", "message"])

print(df.head())
print(df["label"].value_counts())


# ==============================
# 3. CONVERT LABELS
# ham = 0, spam = 1
# ==============================
df["label"] = df["label"].map({"ham": 0, "spam": 1})


# ==============================
# 4. TRAIN-TEST SPLIT
# ==============================
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))


# ==============================
# 5. TOKENIZATION
# ==============================
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print("\nExample text:")
print(X_train.iloc[0])

print("\nConverted to numbers:")
print(X_train_seq[0])


# ==============================
# 6. PADDING
# ==============================
max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

print("\nPadded train shape:", X_train_pad.shape)
print("Padded test shape :", X_test_pad.shape)


# ==============================
# 7. BUILD CNN MODEL
# ==============================
model = Sequential([
    Embedding(input_dim=5000, output_dim=32),

    Conv1D(filters=32, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),

    GlobalAveragePooling1D(),

    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()


# ==============================
# 8. COMPILE MODEL
# ==============================
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# ==============================
# 9. TRAIN MODEL
# ==============================
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    validation_data=(X_test_pad, y_test)
)


# ==============================
# 10. EVALUATE MODEL
# ==============================
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("\nTest Accuracy:", accuracy)


# ==============================
# 11. PREDICTIONS
# ==============================
pred_probs = model.predict(X_test_pad)
preds = (pred_probs > 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))

print("\nClassification Report:")
print(classification_report(y_test, preds))

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64
Train size: 4457
Test size : 1115

Example text:
Reply to win £100 weekly! Where will the 2006 FIFA World Cup be held? Send STOP to 87239 to end service

Converted to numbers:
[87, 2, 172, 485, 461, 121, 33, 5, 3857, 3858, 315, 1268, 30, 2603, 67, 82, 2, 1657, 2, 316, 238]

Padded train shape: (4457, 100)
Padded test shape : (1115, 100)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_3      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8658 - loss: 0.3869 - val_accuracy: 0.8655 - val_loss: 0.3427
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9264 - loss: 0.1861 - val_accuracy: 0.9695 - val_loss: 0.0909
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9814 - loss: 0.0648 - val_accuracy: 0.9848 - val_loss: 0.0521
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9859 - loss: 0.0432 - val_accuracy: 0.9857 - val_loss: 0.0442
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9910 - loss: 0.0320 - val_accuracy: 0.9874 - val_loss: 0.0395
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9874 - loss: 0.0395

Test Accuracy: 0.9874439239501953
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

Confusion Matrix:
[[964   2]
 [ 12 137]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.92      0

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.metrics import classification_report, confusion_matrix

# Build LSTM model
model = Sequential([
    Embedding(input_dim=5000, output_dim=32),
    LSTM(32),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    validation_data=(X_test_pad, y_test)
)

# Evaluate
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("\nTest Accuracy:", accuracy)

# Predict
pred_probs = model.predict(X_test_pad)
preds = (pred_probs > 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))

print("\nClassification Report:")
print(classification_report(y_test, preds))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - accuracy: 0.9118 - loss: 0.2635 - val_accuracy: 0.9785 - val_loss: 0.0765
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.9886 - loss: 0.0493 - val_accuracy: 0.9919 - val_loss: 0.0370
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.9944 - loss: 0.0243 - val_accuracy: 0.9901 - val_loss: 0.0395
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.9971 - loss: 0.0131 - val_accuracy: 0.9919 - val_loss: 0.0361
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.9978 - loss: 0.0091 - val_accuracy: 0.9919 - val_loss: 0.0407
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9919 - loss: 0.0407

Test Accuracy: 0.9919282793998718
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step

Confusion Matrix:
[[962   4]
 [  5 144]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       966
           1       0.97      0.97  

In [9]:
import pickle

# save model
model.save("spam_lstm_model.keras")

# save tokenizer
with open("spam_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# save max_len
with open("spam_config.pkl", "wb") as f:
    pickle.dump({"max_len": max_len}, f)

print("LSTM model, tokenizer, and config saved successfully.")

LSTM model, tokenizer, and config saved successfully.


In [10]:
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# load model
loaded_model = load_model("spam_lstm_model.keras")

# load tokenizer
with open("spam_tokenizer.pkl", "rb") as f:
    loaded_tokenizer = pickle.load(f)

# load config
with open("spam_config.pkl", "rb") as f:
    config = pickle.load(f)

max_len = config["max_len"]

# custom message
sample_text = ["Congratulations! You have won a free prize. Click now."]

# tokenize and pad
sample_seq = loaded_tokenizer.texts_to_sequences(sample_text)
sample_pad = pad_sequences(sample_seq, maxlen=max_len)

# predict
pred_prob = loaded_model.predict(sample_pad)
pred_label = 1 if pred_prob[0][0] > 0.5 else 0

print("Spam probability:", pred_prob[0][0])
print("Predicted label :", pred_label)
print("Prediction      :", "spam" if pred_label == 1 else "ham")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 395ms/step
Spam probability: 0.99892384
Predicted label : 1
Prediction      : spam
